# 🔬 Arm 3 (Priority 3 — Structural Syntax): AST-Guided Policy Optimization (AST-RL)
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs  
**Literature:** TreeDiff (ASE 2025) · VeriSeek (ICSE 2025)  

---

## 🎯 Core Concept
Standard RLVR with binary rewards ($R \in \{0, 1\}$) can still suffer from **lexical overfitting**: a model memorizes variable names and code surface without internalizing the control-flow logic.

**AST-RL** integrates the Python **Abstract Syntax Tree (AST)** into the reward loop:
$$\mathcal{R}_{\text{AST}}(y) = \exp \left( - \alpha \cdot \text{TreeDist}(\text{AST}(y), \text{AST}(y^*)) \right)$$

| Advantage | Limitation |
|---|---|
| Prevents lexical surface overfitting | Misses high-level semantic reframing |
| Structural correctness even with renamed vars | Doesn't capture narrative or creative context |

> See Table 1 §3.4 in the research proposal for the full taxonomy comparison.

In [ ]:
import os
import sys
import ast
import numpy as np
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("."))

print("✅ Arm 3 (AST-RL) Environment Initialized.")

---
## 1. AST Normalizer & Structural Similarity Engine (`simAST`)

In [ ]:
# Canonical implementation imported from src.arms.arm3_ast_rl
from src.arms.arm3_ast_rl import ASTNormalizer, get_ast_signature, simAST, ast_reward, ASTRewardEngine

reward_engine = ASTRewardEngine(beta=0.3, alpha_tree=0.05)
print("✅ Canonical ASTNormalizer, simAST, ast_reward, and ASTRewardEngine imported from src.arms.arm3_ast_rl.")


---
## 2. Invariance Simulation: Same Logic, Different Surface

In [ ]:
ref_code = """
def filter_evens(numbers):
    result = []
    for n in numbers:
        if n % 2 == 0:
            result.append(n)
    return result
"""

# Same algorithm, completely different variable names — AST should be near-identical
renamed_code = """
def get_even_items(elements):
    collected = []
    for item in elements:
        if item % 2 == 0:
            collected.append(item)
    return collected
"""

# Completely different algorithm (quicksort) — AST should differ sharply
decoy_code = """
def sort_items(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    left  = [x for x in arr[1:] if x < pivot]
    right = [x for x in arr[1:] if x >= pivot]
    return sort_items(left) + [pivot] + sort_items(right)
"""

sim_renamed = simAST(renamed_code, ref_code)
sim_decoy   = simAST(decoy_code, ref_code)
r_renamed   = ast_reward(renamed_code, ref_code)
r_decoy     = ast_reward(decoy_code, ref_code)

print(f"simAST (Renamed vs Ref) : {sim_renamed * 100:.1f}%  → R_AST = {r_renamed:.3f}")
print(f"simAST (Decoy vs Ref)   : {sim_decoy * 100:.1f}%  → R_AST = {r_decoy:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=140)
# Similarity bar
axes[0].bar(['Renamed\n(Valid Logic)', 'Decoy\n(Wrong Logic)'], [sim_renamed, sim_decoy],
            color=['#28A745', '#DC3545'], edgecolor='black')
axes[0].set_title('simAST Similarity Score', fontweight='bold')
axes[0].set_ylim(0, 1.15)
# Reward bar
axes[1].bar(['Renamed\n(Valid Logic)', 'Decoy\n(Wrong Logic)'], [r_renamed, r_decoy],
            color=['#28A745', '#DC3545'], edgecolor='black')
axes[1].set_title('AST Reward: R_AST = exp(-α·TreeDist)', fontweight='bold')
axes[1].set_ylim(0, 1.15)
for ax in axes:
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/arm3_ast_similarity.png', dpi=140)
plt.show()
print("✅ AST-RL reward correctly rewards structural equivalence regardless of variable names!")

---
## 3. Full Production AST-RL Policy Optimization (500 Steps — Model M5)

Executes 500-step policy optimization with composite AST structural similarity reward:
$$\mathcal{R}(y) = \mathcal{R}_{\text{exec}} + 0.3 \cdot \text{simAST}(\text{AST}(y), \text{AST}(y^*))$$
Saves LoRA adapter checkpoint to: `checkpoints/rlvr_ast_final`.


In [ ]:
from src.arms.arm3_ast_rl.trainer import ASTRLTrainer
from src.evaluation.registry import BenchmarkRegistry

# Load training pool (HumanEval + EvoEval reference solutions)
registry = BenchmarkRegistry()
tasks = [
    {
        "prompt": t.prompt,
        "test": t.test,
        "entry_point": t.entry_point,
        "canonical_solution": t.canonical_solution,
    }
    for t in registry.get_tasks("L0") + registry.get_tasks("L1")
]
print(f"Loaded {len(tasks)} tasks with canonical reference solutions for AST-RL.")

trainer = ASTRLTrainer(
    output_dir="checkpoints/rlvr_ast_final",
    group_size=4,
    learning_rate=1e-5,
    beta_ast=0.3,
)

NUM_TRAIN_STEPS = 500  # 500 steps production AST-RL run
history = trainer.train(
    tasks=tasks,
    num_steps=NUM_TRAIN_STEPS,
    grad_accum_steps=2,
)

print(f"\n🎉 AST-RL (M5) Training Completed Successfully!")
print(f"   Saved final LoRA adapter to: {trainer.output_dir}")
